In [1]:
# HealthGuard AI - Kidney Disease Data Cleaning
# Author: HealthGuard AI Team
# Date: 2026

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
%matplotlib inline

plt.style.use('seaborn-v0_8')

print("=" * 50)
print("  HealthGuard AI - Kidney Disease Cleaning")
print("=" * 50)
print("Libraries Loaded Successfully!")

  HealthGuard AI - Kidney Disease Cleaning
Libraries Loaded Successfully!


In [2]:
# Load Kidney Disease Dataset

df = pd.read_csv(
    "E:/HealthGuard_AI/data/raw/kidney_disease.csv")

print(f"Original Shape: {df.shape}")
print(f"\nMissing Values:")
print(df.isnull().sum())
df.head()

Original Shape: (400, 26)

Missing Values:
id                  0
age                 9
bp                 12
sg                 47
al                 46
su                 49
rbc               152
pc                 65
pcc                 4
ba                  4
bgr                44
bu                 19
sc                 17
sod                87
pot                88
hemo               52
pcv                70
wc                105
rc                130
htn                 2
dm                  2
cad                 2
appet               1
pe                  1
ane                 1
classification      0
dtype: int64


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [3]:
# Step 1: Clean Target Column
# Remove extra whitespace/tab characters

print("=" * 50)
print("STEP 1: CLEAN TARGET COLUMN")
print("=" * 50)

print("Before cleaning:")
print(df['classification'].value_counts())

df['classification'] = df['classification'].str.strip()

print("\nAfter cleaning:")
print(df['classification'].value_counts())
print("Target column cleaned!")

STEP 1: CLEAN TARGET COLUMN
Before cleaning:
classification
ckd       248
notckd    150
ckd\t       2
Name: count, dtype: int64

After cleaning:
classification
ckd       250
notckd    150
Name: count, dtype: int64
Target column cleaned!


In [4]:
# Step 2: Remove Duplicates

print("=" * 50)
print("STEP 2: REMOVE DUPLICATES")
print("=" * 50)

before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"Before: {before} rows")
print(f"After: {after} rows")
print(f"Removed: {before - after} duplicates")

STEP 2: REMOVE DUPLICATES
Before: 400 rows
After: 400 rows
Removed: 0 duplicates


In [5]:
# Step 3: Handle Missing Values
# Kidney dataset has MANY missing values
# Numeric - median, Categorical - mode

print("=" * 50)
print("STEP 3: HANDLE MISSING VALUES")
print("=" * 50)

print("Before fixing:")
print(f"Total Missing: {df.isnull().sum().sum()}")

# Numeric columns - median
numeric_cols = df.select_dtypes(
    include=['float64', 'int64']).columns
df[numeric_cols] = df[numeric_cols].fillna(
    df[numeric_cols].median())

# Categorical columns - mode
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("\nAfter fixing:")
print(f"Total Missing: {df.isnull().sum().sum()}")
print("All Missing Values Fixed!")

STEP 3: HANDLE MISSING VALUES
Before fixing:
Total Missing: 1009

After fixing:
Total Missing: 0
All Missing Values Fixed!


In [6]:
# Step 4: Encode Target Column
# ckd=1, notckd=0

print("=" * 50)
print("STEP 4: ENCODE TARGET COLUMN")
print("=" * 50)

df['classification'] = df['classification'].map(
    {'ckd': 1, 'notckd': 0})

print("Classification Encoded:")
print(df['classification'].value_counts())
print("ckd=1, notckd=0")

STEP 4: ENCODE TARGET COLUMN
Classification Encoded:
classification
1    250
0    150
Name: count, dtype: int64
ckd=1, notckd=0


In [7]:
# Step 5: Encode Categorical Features

print("=" * 50)
print("STEP 5: ENCODE CATEGORICAL FEATURES")
print("=" * 50)

le = LabelEncoder()

cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))
    print(f" {col} encoded")

print("\nEncoding Complete!")

STEP 5: ENCODE CATEGORICAL FEATURES
 rbc encoded
 pc encoded
 pcc encoded
 ba encoded
 pcv encoded
 wc encoded
 rc encoded
 htn encoded
 dm encoded
 cad encoded
 appet encoded
 pe encoded
 ane encoded

Encoding Complete!


In [8]:
# Step 6: Remove Outliers
# Kidney dataset is small - use gentle approach
# Only remove extreme outliers (2.5 IQR instead of 1.5)

print("=" * 50)
print("STEP 6: REMOVE OUTLIERS")
print("=" * 50)

before = len(df)

outlier_cols = ['age', 'bp', 'bgr', 'bu',
                'sc', 'sod', 'pot', 'hemo', 'pcv']

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 2.5 * IQR  # 2.5 use kiya 1.5 ki jagah
    upper = Q3 + 2.5 * IQR  # 2.5 use kiya 1.5 ki jagah

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {outliers} outliers removed")

    df = df[(df[col] >= lower) & (df[col] <= upper)]

after = len(df)
print(f"\nBefore: {before} rows")
print(f"After: {after} rows")
print(f"Total Removed: {before - after}")

STEP 6: REMOVE OUTLIERS
age: 0 outliers removed
bp: 6 outliers removed
bgr: 26 outliers removed
bu: 25 outliers removed
sc: 27 outliers removed
sod: 4 outliers removed
pot: 0 outliers removed
hemo: 0 outliers removed
pcv: 2 outliers removed

Before: 400 rows
After: 310 rows
Total Removed: 90


In [9]:
# Step 7: Feature Scaling

print("=" * 50)
print("STEP 7: FEATURE SCALING")
print("=" * 50)

X = df.drop('classification', axis=1)
y = df['classification']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("Scaling Complete!")
print(f"Features Shape: {X_scaled.shape}")

STEP 7: FEATURE SCALING
Scaling Complete!
Features Shape: (310, 25)


In [10]:
# Train Test Split FIRST, then SMOTE only on Training data

print("=" * 50)
print("TRAIN TEST SPLIT (BEFORE SMOTE)")
print("=" * 50)

X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y)

print(f"Train (before SMOTE): {len(X_train_raw)}")
print(f"Test (untouched): {len(X_test)}")

print("\n" + "=" * 50)
print("SMOTE ON TRAINING DATA ONLY")
print("=" * 50)

print("Before SMOTE:")
print(f"CKD (1): {(y_train_raw==1).sum()}")
print(f"Normal (0): {(y_train_raw==0).sum()}")

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train_raw, y_train_raw)

print("\nAfter SMOTE:")
print(f"CKD (1): {(y_train==1).sum()}")
print(f"Normal (0): {(y_train==0).sum()}")
print(f"Final Training: {len(X_train)}, Final Testing: {len(X_test)}")

TRAIN TEST SPLIT (BEFORE SMOTE)
Train (before SMOTE): 248
Test (untouched): 62

SMOTE ON TRAINING DATA ONLY
Before SMOTE:
CKD (1): 128
Normal (0): 120

After SMOTE:
CKD (1): 128
Normal (0): 128
Final Training: 256, Final Testing: 62


In [11]:
# Step 10: Save All Cleaned Data

print("=" * 50)
print("STEP 10: SAVE CLEANED DATA")
print("=" * 50)

df_cleaned = pd.concat([X_scaled,
                        y.reset_index(drop=True)], axis=1)
df_cleaned.to_csv(
    "E:/HealthGuard_AI/data/processed/kidney_cleaned_final.csv",
    index=False)

X_train.to_csv(
    "E:/HealthGuard_AI/data/processed/kidney_X_train.csv",
    index=False)
X_test.to_csv(
    "E:/HealthGuard_AI/data/processed/kidney_X_test.csv",
    index=False)
y_train.to_csv(
    "E:/HealthGuard_AI/data/processed/kidney_y_train.csv",
    index=False)
y_test.to_csv(
    "E:/HealthGuard_AI/data/processed/kidney_y_test.csv",
    index=False)

print("Files Saved:")
print(" kidney_cleaned_final.csv")
print(" kidney_X_train.csv")
print(" kidney_X_test.csv")
print(" kidney_y_train.csv")
print(" kidney_y_test.csv")

STEP 10: SAVE CLEANED DATA
Files Saved:
 kidney_cleaned_final.csv
 kidney_X_train.csv
 kidney_X_test.csv
 kidney_y_train.csv
 kidney_y_test.csv


In [12]:
# Cleaning Summary

print("=" * 60)
print("   KIDNEY DISEASE CLEANING - SUMMARY REPORT")
print("=" * 60)

print("\nTECHNIQUES USED:")
print("-" * 40)
print("1. Target Column Cleaning (str.strip)")
print("2. Duplicate Removal")
print("3. Missing Values - Median & Mode Imputation")
print("4. Target Encoding (ckd=1, notckd=0)")
print("5. Label Encoding - Categorical Features")
print("6. IQR Outlier Removal")
print("7. Standard Scaling")
print("8. SMOTE")
print("9. Train Test Split 80/20")

print(f"\nOriginal Rows: 400")
print(f"After Cleaning: {len(df)}")
print(f"Training Set (after SMOTE): {len(X_train)}")
print(f"Testing Set (untouched, real): {len(X_test)}")

print("\nKidney Disease Cleaning Complete!")
print("=" * 60)

   KIDNEY DISEASE CLEANING - SUMMARY REPORT

TECHNIQUES USED:
----------------------------------------
1. Target Column Cleaning (str.strip)
2. Duplicate Removal
3. Missing Values - Median & Mode Imputation
4. Target Encoding (ckd=1, notckd=0)
5. Label Encoding - Categorical Features
6. IQR Outlier Removal
7. Standard Scaling
8. SMOTE
9. Train Test Split 80/20

Original Rows: 400
After Cleaning: 310
Training Set (after SMOTE): 256
Testing Set (untouched, real): 62

Kidney Disease Cleaning Complete!


In [13]:
# Save Scaler for Web App
import pickle

scaler_path = "E:/HealthGuard_AI/models/saved/kidney_scaler.pkl"
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(" Kidney Disease Scaler Saved!")
print(f"Location: {scaler_path}")

 Kidney Disease Scaler Saved!
Location: E:/HealthGuard_AI/models/saved/kidney_scaler.pkl


In [14]:
# FIXED: Pipeline on RAW unscaled data + drop id
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import pickle
import pandas as pd
import numpy as np

# Reload RAW kidney data
df_kidney = pd.read_csv("E:/HealthGuard_AI/data/raw/kidney_disease.csv")

# Clean target column
df_kidney['classification'] = df_kidney['classification'].str.strip()
df_kidney['classification'] = df_kidney['classification'].map(
    {'ckd': 1, 'notckd': 0})

# Drop id column
if 'id' in df_kidney.columns:
    df_kidney = df_kidney.drop('id', axis=1)

# Fill missing values
le = LabelEncoder()
num_cols = df_kidney.select_dtypes(include=['float64','int64']).columns
df_kidney[num_cols] = df_kidney[num_cols].fillna(
    df_kidney[num_cols].median())
cat_cols = df_kidney.select_dtypes(include=['object']).columns
for col in cat_cols:
    df_kidney[col] = df_kidney[col].fillna(df_kidney[col].mode()[0])
    df_kidney[col] = le.fit_transform(df_kidney[col].astype(str))

X_raw = df_kidney.drop('classification', axis=1)
y_raw = df_kidney['classification']

# SMOTE on raw data
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_raw, y_raw)

# Split
X_tr, X_te, y_tr, y_te = train_test_split(
    X_resampled, y_resampled,
    test_size=0.3, random_state=42,
    stratify=y_resampled)

feature_names = X_raw.columns.tolist()
print("Kidney Features:", feature_names)

# Pipeline handles scaling
kidney_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(
        n_estimators=100, random_state=42, max_depth=5))
])

kidney_pipeline.fit(X_tr, y_tr)
y_pred = kidney_pipeline.predict(X_te)
print(f"Accuracy: {accuracy_score(y_te, y_pred):.4f}")

# High risk test
test_sample = pd.DataFrame([[
    55, 160, 1.020, 0, 0, 1, 1, 0, 0,
    250, 30, 3.5, 140, 4.5, 10, 40,
    8000, 4.5, 1, 1, 0, 1, 0, 0
]], columns=feature_names)
prob = kidney_pipeline.predict_proba(test_sample)[0][1]
print(f"High Risk Test: {prob * 100:.2f}%")

pipeline_path = "E:/HealthGuard_AI/models/saved/kidney_pipeline.pkl"
with open(pipeline_path, 'wb') as f:
    pickle.dump(kidney_pipeline, f)
print("Kidney Pipeline Saved!")
print("Features:", feature_names)

Kidney Features: ['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']
Accuracy: 0.9933
High Risk Test: 86.67%
Kidney Pipeline Saved!
Features: ['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']
